## Feature generation

In [ ]:
import pandas as pd
import numpy as np
import re
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, EState, GraphDescriptors, MolSurf
import skfp.fingerprints as sfp
from modules.data_loader import *

In [ ]:
visco = load_cleaned()

In [ ]:
visco.info()

In [ ]:
visco.head()

In [ ]:
smiles = visco['smiles_surfactant1']
smiles = pd.concat([smiles, visco['smiles_surfactant2']])
smiles = pd.concat([smiles, visco['smiles_additive1']])
smiles = pd.concat([smiles, visco['smiles_additive2']])
smiles = smiles.drop_duplicates()
smiles = smiles.dropna()
len(smiles)

In [ ]:
# Tail length
def extract_tail_c_number(smiles: str):
    # For now just count n of leading C's
    if pd.isna(smiles):
        return np.nan
    n_c = re.match(r"^(C+)", smiles)
    return len(n_c.group(1)) if n_c else 0


# Surfactant type is already in the data but further more advance type can be added later(look in README)

def extract_molecular_weight(smiles: str):
    if pd.isna(smiles):
        return np.nan
    mol = Chem.MolFromSmiles(smiles)
    return ExactMolWt(mol)


def generate_chen(smiles: str):
    if pd.isna(smiles):
        return pd.Series({
            "chen_MolLogP": np.nan,
            "chen_Chi1v": np.nan,
            "chen_VSA_EState7": np.nan,
            "chen_VSA_EState8": np.nan,
            "chen_qed": np.nan,
            "chen_FpDensityMorgan1": np.nan,
            "chen_SMR_VSA5": np.nan,
            "chen_PEOE_VSA6": np.nan,
            "chen_MinPartialCharge": np.nan,
            "chen_BertzCT": np.nan,
            "chen_VSA_EState1": np.nan,
            "chen_MaxAbsEStateIndex": np.nan,
            "chen_Balaban": np.nan,
            "chen_SlogP_VSA2": np.nan,
            "chen_HallKierAlpha": np.nan,
            "chen_EState_VSA5": np.nan,
            "chen_MaxPartialCharge": np.nan,
            "chen_PEOE_VSA7": np.nan,
            "chen_PEOE_VSA8": np.nan
        })

    mol = Chem.MolFromSmiles(smiles)

    return pd.Series({
        "chen_MolLogP": Descriptors.MolLogP(mol),
        "chen_Chi1v": GraphDescriptors.Chi1v(mol),
        "chen_VSA_EState7": EState.EState_VSA.EState_VSA7(mol),
        "chen_VSA_EState8": EState.EState_VSA.EState_VSA8(mol),
        "chen_qed": QED.qed(mol),
        "chen_FpDensityMorgan1": Descriptors.FpDensityMorgan1(mol),
        "chen_SMR_VSA5": MolSurf.SMR_VSA5(mol),
        "chen_PEOE_VSA6": MolSurf.PEOE_VSA6(mol),
        "chen_MinPartialCharge": Descriptors.MinPartialCharge(mol),
        "chen_BertzCT": GraphDescriptors.BertzCT(mol),
        "chen_VSA_EState1": EState.EState_VSA.EState_VSA1(mol),
        "chen_MaxAbsEStateIndex": EState.MaxAbsEStateIndex(mol),
        "chen_Balaban": GraphDescriptors.BalabanJ(mol),
        "chen_SlogP_VSA2": MolSurf.SlogP_VSA2(mol),
        "chen_HallKierAlpha": Descriptors.HallKierAlpha(mol),
        "chen_EState_VSA5": EState.EState_VSA.EState_VSA5(mol),
        "chen_MaxPartialCharge": Descriptors.MaxPartialCharge(mol),
        "chen_PEOE_VSA7": MolSurf.PEOE_VSA7(mol),
        "chen_PEOE_VSA8": MolSurf.PEOE_VSA8(mol)
    })

# Requires .transform() - due  to representation it's not really suitable to put this into pandas(numpy better?)
# features = sfp.PhysiochemicalPropertiesFingerprint(fp_size=256)

In [ ]:
expert_chen = pd.DataFrame(smiles, columns=['smiles'])

In [ ]:
expert_chen.head()

In [ ]:
chen_columns = [
    "chen_MolLogP",
    "chen_Chi1v",
    "chen_VSA_EState7",
    "chen_VSA_EState8",
    "chen_qed",
    "chen_FpDensityMorgan1",
    "chen_SMR_VSA5",
    "chen_PEOE_VSA6",
    "chen_MinPartialCharge",
    "chen_BertzCT",
    "chen_VSA_EState1",
    "chen_MaxAbsEStateIndex",
    "chen_Balaban",
    "chen_SlogP_VSA2",
    "chen_HallKierAlpha",
    "chen_EState_VSA5",
    "chen_MaxPartialCharge",
    "chen_PEOE_VSA7",
    "chen_PEOE_VSA8"
]

In [ ]:
expert_chen[chen_columns] = expert_chen.apply(
    lambda row: generate_chen(row['smiles']),
    axis=1,
    result_type='expand'
)

expert_chen['molecular_weight'] = expert_chen['smiles'].apply(extract_molecular_weight)
expert_chen['tail_length'] = expert_chen['smiles'].apply(extract_tail_c_number)

In [ ]:
save_features(expert_chen, 'expert_chen')